# Spike Exploratory Statistics

Barcode/mutation background statistics and replicate functional score correlations.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import warnings
from functools import reduce

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]
seed = cfg["seed"]
output_dir = spike["output_dir"]
condition_titles = spike.get("condition_titles", {c: c for c in spike["experiment_conditions"]})

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline

plt.rcParams.update({"legend.frameon": False, "font.size": 11, "font.weight": "normal"})

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})

# Optionally subsample
subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=seed)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )

print(f"Loaded {len(func_score_df)} variants")

## Variant statistics

In [ ]:
for group, group_df in func_score_df.groupby(["condition", "replicate"]):
    print(f"{group[0]} - rep {group[1]}: {round(group_df.n_subs.mean(), 5)} subs/variant, {len(group_df.aa_substitutions.unique())} unique mutations")

## Barcode and background distributions

In [ ]:
saveas = "raw_data_summary_barcodes_backgrounds_hist"
fig, ax = plt.subplots(2, 3, sharex="row", sharey="row", figsize=[6.4, 5.5])

row = 0
for col, (condition, condition_df) in enumerate(func_score_df.groupby("condition")):
    iter_ax = ax[row, col]
    mut_df_rep = condition_df.query("aa_substitutions != ''").assign(
        num_muts=[len(aa_subs.split()) for aa_subs in condition_df.query("aa_substitutions != ''").aa_substitutions]
    )
    sns.histplot(mut_df_rep.query("num_muts <= 10"), x="num_muts", ax=iter_ax, hue="replicate", discrete=True)
    for rep, rep_df in mut_df_rep.groupby("replicate"):
        iter_ax.axvline(rep_df["num_muts"].mean(), linestyle=("-" if rep == 1 else "--"))
    if col != 2:
        iter_ax.get_legend().remove()
    n1 = len(mut_df_rep.query("replicate == 1")) // 1000
    n2 = len(mut_df_rep.query("replicate == 2")) // 1000
    iter_ax.text(0.1, 1.1, f"$N={n1}K, {n2}K$", ha="left", va="top", transform=iter_ax.transAxes)
    iter_ax.set_xlabel("number of amino-acid substitutions per variant" if col == 1 else "")
    iter_ax.set_ylabel("variant counts" if col == 0 else "")
    iter_ax.set_xticks(range(1, 11), labels=range(1, 11), ha="center", size=7)
    sns.despine(ax=iter_ax)
    iter_ax.set_title(condition_titles.get(condition, condition), y=1.15)

row = 1
collapsed_bc_df = func_score_df.groupby(["replicate", "condition", "aa_substitutions"]).aggregate("mean").reset_index()
for col, (condition, condition_df) in enumerate(collapsed_bc_df.groupby("condition")):
    iter_ax = ax[row, col]
    mut_df_rep = pd.DataFrame()
    for rep, rep_df in condition_df.groupby("replicate"):
        times_seen = rep_df["aa_substitutions"].str.split().explode().value_counts()
        if (times_seen == times_seen.astype(int)).all():
            times_seen = times_seen.astype(int)
        times_seen = pd.DataFrame(times_seen)
        times_seen.index.name = "mutation"
        mut_df_rep = pd.concat([mut_df_rep, times_seen.assign(replicate=rep).reset_index()])
    sns.histplot(mut_df_rep.query("count <= 50"), x="count", ax=iter_ax, element="step", hue="replicate", discrete=True)
    for rep, rep_df in mut_df_rep.groupby("replicate"):
        iter_ax.axvline(rep_df["count"].mean(), linestyle=("-" if rep == 1 else "--"))
    iter_ax.get_legend().remove()
    n1 = len(mut_df_rep.query("replicate == 1"))
    n2 = len(mut_df_rep.query("replicate == 2"))
    iter_ax.text(0.1, 1.1, f"$N={n1}, {n2}$", ha="left", va="top", transform=iter_ax.transAxes)
    iter_ax.set_xlabel("number of variant backgrounds\nfor a given amino-acid substitution" if col == 1 else "")
    iter_ax.set_ylabel("mutation counts" if col == 0 else "")
    xticks = [i for i in range(0, 51) if i % 5 == 0]
    iter_ax.set_xticks(xticks, labels=xticks, ha="center", size=7)
    sns.despine(ax=iter_ax)

plt.tight_layout()
ax[0, 0].text(-0.1, 1.06, "A", ha="right", va="bottom", size=15, weight="bold", transform=ax[0, 0].transAxes)
ax[1, 0].text(-0.1, 1.06, "B", ha="right", va="bottom", size=15, weight="bold", transform=ax[1, 0].transAxes)
fig.subplots_adjust(hspace=0.6)
fig.savefig(f"{output_dir}/{saveas}.pdf")
plt.show()

## Replicate functional score correlation

In [ ]:
saveas = "replicate_functional_score_correlation_scatter"
fig, ax = plt.subplots(2, 3, sharex="row", sharey="row", figsize=[6.4, 5.3])
collapsed_bc_df = (
    func_score_df.groupby(["replicate", "condition", "aa_substitutions"])
    .aggregate("mean")
    .reset_index()
    .assign(is_stop=lambda x: x["aa_substitutions"].str.contains(r"\*", regex=True))
)

lim = [-3.8, 2.8]
ticks = np.linspace(-3, 2, 6)
for col, (condition, condition_df) in enumerate(collapsed_bc_df.groupby("condition")):
    # Scatter plot
    iter_ax = ax[0, col]
    mut_df_rep = reduce(
        lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how="inner"),
        [
            rep_df.rename({"func_score": f"rep_{rep}_func_score"}, axis=1).set_index("aa_substitutions")
            for rep, rep_df in condition_df.groupby("replicate")
        ],
    )
    mut_df_rep = mut_df_rep.assign(
        is_stop=lambda x: x.index.str.contains(r"\*", regex=True),
        n_subs=lambda x: [len(a.split()) for a in x.index.values],
    )
    for istp, color in zip([False, True], ["darkgrey", "red"]):
        sns.scatterplot(
            mut_df_rep.query("is_stop == @istp"),
            x="rep_1_func_score", y="rep_2_func_score",
            ax=iter_ax, c=color, alpha=0.5 if istp else 0.2, legend=False,
        )
    iter_ax.plot([-3.5, 2.5], [-3.5, 2.5], "--", lw=2, c="royalblue")
    iter_ax.set_ylim(lim)
    iter_ax.set_xlim(lim)
    if col == 0:
        iter_ax.set_yticks(ticks, labels=ticks)
    iter_ax.set_xticks(ticks, labels=ticks, rotation=90)
    corr = pearsonr(mut_df_rep["rep_1_func_score"], mut_df_rep["rep_2_func_score"])[0]
    iter_ax.annotate(f"$r = {corr:.2f}$", (0.1, 0.9), xycoords="axes fraction", fontsize=12)
    iter_ax.set_title(condition_titles.get(condition, condition))
    sns.despine(ax=iter_ax)

    # Violin plot
    iter_ax = ax[1, col]
    sns.violinplot(
        condition_df, x="is_stop", y="func_score", hue="replicate",
        split=True, gap=0.1, inner="quart", palette=["0.5", "0.75"], ax=iter_ax,
    )
    sns.despine(ax=iter_ax)
    if col != 2:
        iter_ax.get_legend().remove()
    else:
        iter_ax.legend(bbox_to_anchor=(1.25, 1.05), title="replicate")
    if col == 0:
        iter_ax.set_yticks(ticks, labels=ticks)

ax[0, 0].set_xlabel("")
ax[0, 0].set_ylabel("replicate 2\n functional score")
ax[0, 1].set_xlabel("replicate 1 functional score")
ax[0, 2].set_xlabel("")
ax[1, 0].set_xlabel("")
ax[1, 0].set_ylabel("functional score")
ax[1, 1].set_xlabel("variants contain stop codon mutations")
ax[1, 1].set_ylabel("")
ax[1, 2].set_xlabel("")
ax[1, 2].set_ylabel("")
ax[0, 0].text(-0.1, 1.06, "A", ha="right", va="bottom", size=15, weight="bold", transform=ax[0, 0].transAxes)
ax[1, 0].text(-0.1, 1.06, "B", ha="right", va="bottom", size=15, weight="bold", transform=ax[1, 0].transAxes)
plt.tight_layout()
fig.subplots_adjust(wspace=0.08, hspace=0.5)
fig.savefig(f"{output_dir}/{saveas}.pdf")
plt.show()